# GA4 ecommerce funnel analysis

**Business question:** At which stage are the largest user losses occurring, which segments contribute most to the missed purchase opportunity, and what should the product team investigate first?

## 1. Data scope

The public GA4 sample covers 1 Nov 2020–31 Jan 2021. `add_to_cart` is missing on 18 dates, including 21–24 Nov, so headline metrics use the stable 25 Nov 2020–31 Jan 2021 window.

In [1]:
import pandas as pd

segments = pd.read_csv('../data/source/funnel_segments.csv')
funnel = pd.read_csv('../data/processed/funnel.csv')
categories = pd.read_csv('../data/processed/category_opportunity.csv')
weekly = pd.read_csv('../data/processed/weekly_funnel.csv')
print(segments.loc[segments.segment_type == 'Overall'].to_string(index=False))

segment_type segment_value  view_sessions  cart_sessions  checkout_sessions  purchase_sessions  view_to_cart_rate  cart_to_checkout_rate  checkout_to_purchase_rate  view_to_purchase_rate  purchases_outside_ordered_path
     Overall  All sessions          56696          14618               5091               2668             0.2578                 0.3483                     0.5241                 0.0471                             223


## 2. Ordered session funnel

SQL assigns a session to a stage only if events occur in order: `view_item → add_to_cart → begin_checkout → purchase`.

In [2]:
funnel[['stage', 'sessions', 'step_conversion_rate', 'dropoff_sessions', 'dropoff_rate']]

           stage  sessions  step_conversion_rate  dropoff_sessions  dropoff_rate
  Viewed product     56696              1.000000                 0      0.000000
   Added to cart     14618              0.257831             42078      0.742169
Started checkout      5091              0.348269              9527      0.651731
       Purchased      2668              0.524062              2423      0.475938


**Finding:** 42,078 sessions, or 74.2%, are lost between product view and add to cart. This is the first area to diagnose.

## 3. Segment priority

In [3]:
overall_rate = segments.loc[segments.segment_type == 'Overall', 'view_to_purchase_rate'].iloc[0]
channels = segments[segments.segment_type == 'First-user acquisition channel'].copy()
channels = channels[channels.segment_value.isin(['google / organic', 'google / cpc', '(direct) / (none)'])]
channels['scenario_extra_purchases'] = ((channels.view_sessions * overall_rate) - channels.purchase_sessions).clip(lower=0).round()
print(channels[['segment_value', 'view_sessions', 'view_to_purchase_rate', 'scenario_extra_purchases']].sort_values('scenario_extra_purchases', ascending=False).to_string(index=False))

    segment_value  view_sessions  view_to_purchase_rate  scenario_extra_purchases
 google / organic          17442                 0.0399                       126
     google / cpc           2320                 0.0375                        22
(direct) / (none)          13006                 0.0463                        11


**Finding:** Google organic has the largest actionable volume-adjusted gap: 17,442 viewed sessions and 3.99% purchase conversion versus 4.71% overall. Reaching the overall rate is a transparent scenario of about 126 additional purchases, not a forecast.

## 4. Product category opportunity

In [4]:
print(categories.head(6).to_string(index=False))

           product_category  view_sessions  cart_sessions  view_to_cart_rate  overall_benchmark_rate  scenario_extra_carts  scenario_extra_purchases
Home/Shop by Brand/YouTube/           3620            715             0.1975                  0.2578                   218                        40
                       Bags           2018            356             0.1764                  0.2578                   164                        30
            Lifestyle/Bags/           3543            814             0.2297                  0.2578                    99                        18
                 Backpacks/            361             17             0.0471                  0.2578                    76                        14
            Men's T-Shirts/           1344            277             0.2061                  0.2578                    70                        13
              Home/Apparel/           5001           1223             0.2446                  0.2578      

Category analysis stops at add to cart because later item-category values are not comparable. The scenario uses the overall view-to-cart rate and downstream cart-to-purchase rate.

## 5. Validation

In [5]:
print(checks.to_string(index=False))

                      check  passed
     Funnel counts decrease    True
 Weekly purchases reconcile    True
Ordered purchases reconcile    True


## 6. Recommendation

Investigate product-detail and add-to-cart friction first. Begin with Google organic sessions and the YouTube and Bags category paths. Add page-level diagnostics such as stock status, price visibility, CTA exposure, errors, and load time; then test the most supported product hypothesis with an A/B experiment. Device rates are similar, so device is not the first priority.

This observational sample identifies where to investigate; it does not prove causality.